In [1]:
import pandas as pd

import random

import os

from datetime import datetime, timedelta

In [2]:
random.seed(42)

In [5]:
# ─────────────────────────────────────────────────────────────

# 1. PRODUCT CATALOGUE

# ─────────────────────────────────────────────────────────────

categories = {

'Kitchen Appliances': [

'TurboBlend Pro 5000', 'InstaBrew Coffee Maker',

'CrispAir Fryer XL', 'SliceMaster Food Processor',

'SteamKing Rice Cooker'

],

'Bedroom Furniture': [

'DreamRest Memory Foam Mattress', 'CloudPillow Set (2-pack)',

'NightOwl LED Bedside Lamp', 'WoodCraft Bed Frame Queen',

'VelvetTouch Duvet Cover'

],

'Living Room': [

'SoundBar X200 Bluetooth', 'SmartTV 55in 4K UHD',

'CozySofa L-Shape Sectional', 'AirPurifier ProMax',

'GlowLight Floor Lamp'

],

'Bathroom': [

'AquaFlow Rainfall Showerhead', 'MirrorPro LED Vanity Mirror',

'SoftTouch Luxury Towel Set', 'FoamPure Electric Toothbrush',

'AromaSpa Diffuser'

],
}

In [6]:
positive_templates = [

'Absolutely love my {product}! Works perfectly and arrived on time. Would buy again.',

'Great product. The {product} exceeded my expectations in every way. Highly recommend!',

'Five stars. The {product} is well built, easy to set up, and my whole family loves it.',

'I was skeptical at first but the {product} is genuinely impressive. Fast delivery too.',

'Best purchase I made this year. The {product} works exactly as advertised. Very happy.',

'Solid quality and great value. The {product} feels premium and performs even better.',

'My {product} arrived well packaged and started working immediately. No issues at all.',

'Bought the {product} as a gift and the recipient loved it. Excellent build quality.',

'The {product} is easy to clean, quiet during operation, and looks great. 5 stars!',

'Outstanding product. The {product} has made my daily routine so much easier. Thank you!',

]

In [7]:
negative_templates = [

'Terrible quality. My {product} broke after just 2 weeks of normal use!!!',

'DO NOT BUY. The {product} stopped working completely. Customer service was useless.',

'Very disappointed. The {product} arrived damaged and smells like chemicals. Returning it.',

'Waste of money. The {product} motor makes a horrible grinding noise from day one.',

'The {product} overheated and stopped working after 3 uses. Safety hazard. 1 star.',

'Poorly made. The {product} plastic cracked within a week. Cheaply built garbage.',

'I wanted to love this but the {product} died in week one. Warranty claim was denied.',

'The {product} leaks, smells like burnt plastic, and the manual is useless. Awful.',

'Sent the wrong item twice. When the {product} finally arrived it was defective. Never again.',

'The {product} worked for exactly 4 days then completely stopped. Total waste of money.',

]

In [8]:
neutral_templates = [

'The {product} is okay. Does what it says but nothing special. Average quality.',

'Decent product for the price. The {product} has some quirks but generally works fine.',

'Mixed feelings about the {product}. Setup was easy but performance is mediocre.',

'The {product} arrived on time and works as described. Not great not terrible.',

'It is a fine product. The {product} gets the job done but I expected better quality.',

'The {product} is functional but the instructions are confusing. Took a while to set up.',

]

In [9]:
# Additional templates with specific pain points for richer topic modeling

delivery_templates = [

'The {product} itself is fine but the packaging was completely destroyed on arrival.',

'Delivery took 3 weeks and the {product} box was crushed. Product inside was okay though.',

'Wrong item delivered initially. Replacement {product} finally came after 10 day.',
]

In [10]:
service_templates = [

'The {product} stopped working. Customer support took 2 weeks to respond. Very frustrated.',

'Had an issue with my {product} but the support team resolved it quickly. Happy with the service.',

'Trying to return my {product} has been a nightmare. No response from customer service.',

]

In [11]:
safety_templates = [

'WARNING: My {product} started smoking after 10 minutes of use. Possible fire hazard!',

'The {product} emits a strange chemical smell when running. Cannot use it safely indoors.',

'My {product} sparked when I plugged it in. Extremely dangerous. Reporting to manufacturer.',

]

In [12]:
# ─────────────────────────────────────────────────────────────

# 3. RATING <-> TEMPLATE MAPPING

# Maps star rating to likely template pool + weights

# ─────────────────────────────────────────────────────────────

rating_template_map = {

1: (negative_templates + safety_templates, [0.7]*10 + [0.3]*3),

2: (negative_templates + delivery_templates,[0.6]*10 + [0.4]*3),
    
3: (neutral_templates + delivery_templates, [0.7]*6 + [0.3]*3),

4: (positive_templates + neutral_templates[:2], [0.8]*10 + [0.2]*2),

5: (positive_templates + service_templates[-1:], [0.9]*10 + [0.1]*1),

}

In [14]:
# ─────────────────────────────────────────────────────────────

# 4. NOISE INJECTION

# Simulates real-world messy review text

# ─────────────────────────────────────────────────────────────

def inject_noise(text):

 noise_type = random.random()

 if noise_type < 0.10:

  text = text.upper() # ALL CAPS reviewer

 elif noise_type < 0.18:

  text = text + ' https://fakeurl.com/ref123' # URL in review
 
 elif noise_type < 0.24:

  text = '<p>' + text + '</p>' # HTML artifact

 elif noise_type < 0.30:

  text = text + ' !!!!!!' # Excessive punctuation

 elif noise_type < 0.35:

  text = text.replace(' ', ' ') # Double spaces

 return text

In [19]:
# ─────────────────────────────────────────────────────────────

# 5. MAIN GENERATION LOOP

# ─────────────────────────────────────────────────────────────

def generate_dataset(n=5000):

 records = []

 start_date = datetime(2023, 1, 1)
 # Skewed rating distribution (mirrors real e-commerce data)

 # 5-star reviews are most common, 2-star least common

 rating_weights = {1: 0.10, 2: 0.07, 3: 0.13, 4: 0.25, 5: 0.45}

 ratings = list(rating_weights.keys())

 weights = list(rating_weights.values())
 for i in range(1, n + 1):

  category = random.choice(list(categories.keys()))

  product = random.choice(categories[category])

  rating = random.choices(ratings, weights=weights, k=1)[0]
  # Pick a template pool for this rating

  pool, pool_weights = rating_template_map[rating]  

  # Normalise weights in case pool sizes differ

  norm_w = pool_weights[:len(pool)]

  template = random.choices(pool, weights=norm_w, k=1)[0]
  review_text = template.format(product=product)

  review_text = inject_noise(review_text)  

  # Random date within the past ~2 years

  days_offset = random.randint(0, 730)

  review_date = (start_date + timedelta(days=days_offset)).strftime('%Y-%m-%d')
  records.append({

   'review_id': i,

   'product_category': category,

   'product_name': product,

    'review_text': review_text,

    'star_rating': rating,

    'review_date': review_date,

    'verified_purchase': random.choice([True, True, True, False]),

  })
 return pd.DataFrame(records)  

In [24]:
# ─────────────────────────────────────────────────────────────

# 6. RUN AND SAVE

# ─────────────────────────────────────────────────────────────

if __name__ == '__main__':

 os.makedirs('/raw', exist_ok=True)

 df = generate_dataset(n=5000)

 output_path = 'raw/homenest_reviews.csv'

 df.to_csv(output_path, index=False)
 print(f'Dataset saved to {output_path}')

 print(f'Shape: {df.shape}')

 print()

 print('Rating distribution:')

 print(df['star_rating'].value_counts().sort_index())

 print()

 print('Category distribution:')

 print(df['product_category'].value_counts())

 print()

 print('Sample reviews:')

 print(df[['product_name','star_rating','review_text']].sample(5).to_string())

Dataset saved to raw/homenest_reviews.csv
Shape: (5000, 7)

Rating distribution:
star_rating
1     512
2     352
3     615
4    1202
5    2319
Name: count, dtype: int64

Category distribution:
product_category
Bathroom              1256
Kitchen Appliances    1253
Living Room           1248
Bedroom Furniture     1243
Name: count, dtype: int64

Sample reviews:
                   product_name  star_rating                                                                                              review_text
41     CloudPillow Set (2-pack)            5     My CloudPillow Set (2-pack) arrived well packaged and started working immediately. No issues at all.
2495   CloudPillow Set (2-pack)            4   Outstanding product. The CloudPillow Set (2-pack) has made my daily routine so much easier. Thank you!
4616  NightOwl LED Bedside Lamp            5  Outstanding product. The NightOwl LED Bedside Lamp has made my daily routine so much easier. Thank you!
3436        SmartTV 55in 4K UHD        